# BERT 模型训练
## AI电商评论分析系统 — 情感分析模型微调
本 notebook 用于 BERT-base-chinese 模型的微调训练。

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

from nlp.src.data_processing.preprocessor import ReviewPreprocessor
from nlp.src.models.bert.config import BERTConfig

print('CUDA available:', torch.cuda.is_available())
print('Device count:', torch.cuda.device_count() if torch.cuda.is_available() else 0)

In [ ]:
# 配置参数
config = BERTConfig(
    model_name='bert-base-chinese',
    num_labels=3,
    max_length=128,
    batch_size=16,
    epochs=5,
    learning_rate=2e-5,
    output_dir='../data/models/bert-sentiment'
)
print(config)

In [ ]:
# 加载数据
preprocessor = ReviewPreprocessor()
df = preprocessor.load_csv('../data/raw/train.csv')
df = preprocessor.preprocess_dataframe(df, label_column='label')
train_df, val_df = preprocessor.train_val_split(df, val_ratio=0.2)
print(f'训练集: {len(train_df)}, 验证集: {len(val_df)}')

In [ ]:
# Tokenization
tokenizer = BertTokenizer.from_pretrained(config.model_name)

def tokenize_fn(examples):
    return tokenizer(examples['content'], truncation=True, padding='max_length', max_length=config.max_length)

train_dataset = Dataset.from_pandas(train_df[['content', 'label']]).map(tokenize_fn, batched=True)
val_dataset = Dataset.from_pandas(val_df[['content', 'label']]).map(tokenize_fn, batched=True)

In [ ]:
# 模型初始化
model = BertForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=config.num_labels,
    hidden_dropout_prob=config.hidden_dropout,
)

In [ ]:
# 训练参数
training_args = TrainingArguments(
    output_dir=config.output_dir,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=config.learning_rate,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size * 2,
    num_train_epochs=config.epochs,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=lambda p: {'acc': accuracy_score(p.label_ids, np.argmax(p.predictions, -1)),
                                'f1': f1_score(p.label_ids, np.argmax(p.predictions, -1), average='weighted')},
)

In [ ]:
# 开始训练
trainer.train()
trainer.save_model(config.output_dir)
tokenizer.save_pretrained(config.output_dir)
print(f'Model saved to {config.output_dir}')

In [ ]:
# 最终评估
eval_result = trainer.evaluate()
print('Evaluation results:', eval_result)